# Data Pipeline & Feature Engineering

## Import Datase: for only years 2014-2026

In [32]:
import pandas as pd

# The 2021-2024 file only covers 4 seasons, so use the full 1980-2026 baseline.
# SEASON_ID is the year the season ends (2014 == 2013-14).
df = pd.read_parquet("data/processed/player_seasons_full.parquet")
df = df[df["SEASON_ID"].between(2014, 2026)].reset_index(drop=True)

print(df.shape, df["SEASON_ID"].min(), "-", df["SEASON_ID"].max())
df.head()


(6942, 53) 2014 - 2026


,PLAYER_ID,PLAYER,TEAM_ID,TEAM,AGE,GP,MPG,PTS,REB,AST,...,Z_USG_PCT,Z_PER,Z_WS,Z_WS_48,Z_BPM,Z_VORP,Z_TEAM_WIN_PCT,RTS,CARRY_PROXY,Z_CARRY_PROXY
0,201985,AJ Price,1.610613e+09,MIN,27.0,28.0,3.5,1.6,0.4,0.5,...,0.740161,-1.272808,-1.368911,-2.182786,-1.924489,-0.784961,-0.082559,-7.157292,-5.297551,-1.141223
1,2399,Mike Dunleavy,1.610613e+09,CHI,33.0,82.0,31.5,11.3,4.2,2.3,...,-0.543475,-0.555452,0.809246,0.408068,0.363280,0.476880,0.547423,0.842708,-0.457991,-0.141800
2,201144,Mike Conley,1.610613e+09,MEM,26.0,73.0,33.5,17.2,2.9,6.0,...,1.030015,1.275041,1.378918,1.187272,1.588870,1.672308,0.709790,0.442708,0.455996,0.046949
3,203077,Michael Kidd-Gilchrist,1.610613e+09,CHA,20.0,62.0,24.2,7.2,5.2,0.8,...,-0.916144,-0.703871,-0.330098,-0.039975,-0.821458,-0.718548,0.151249,-2.457292,2.251233,0.417685
4,203487,Michael Carter-Williams,1.610613e+09,PHI,22.0,70.0,34.5,16.7,6.2,6.3,...,1.257757,0.161903,-0.933280,-1.442542,-0.167809,-0.120834,-1.745191,-6.057292,-7.618600,-1.620546


## Merge DARKO DPM (NBA ID, else accent-folded names)

In [ ]:
import data_pipeline as dp
from rebuild_model2 import NAME_ALIASES, SEASONS, merge_darko

# Merge DARKO DPM for 2014-2026 from data/raw/darko_<season>.csv.
# Same logic as rebuild_model2.py (imported, so the notebook and the script can't drift apart):
#   1. Match on DARKO's "NBA ID" column when the file has one (only darko_2027.csv does today).
#   2. Otherwise match on dp.normalize_name within the same season. It folds accents instead of
#      deleting them ("Nikola Jokić" -> "nikola jokic", same as DARKO's "Nikola Jokic") and ignores
#      case, punctuation and Jr./Sr./II/III suffixes.
#   3. NAME_ALIASES covers players DARKO lists under another name (Enes Freedom -> Enes Kanter, ...).
# The old regex (r'[^a-z\s]') deleted accented letters, so "jokić" became "joki" and Jokić / Dončić
# silently got DPM = 0 in every season.
df["PLAYER_ID"] = df["PLAYER_ID"].astype(str)
df = merge_darko(df)

elig = df["ELIGIBLE"] & df["HAS_ADV"]
print(f"DARKO seasons: {SEASONS.start}-{SEASONS.stop - 1}  |  aliases: {len(NAME_ALIASES)}")
print(f"Matched {df['DPM'].notna().sum()} of {len(df)} rows "
      f"({df.loc[elig, 'DPM'].notna().mean():.1%} of eligible, "
      f"{df.loc[df['MVP_SHARE'] > 0, 'DPM'].notna().mean():.1%} of MVP vote-getters)")
unmatched = df[elig & df["DPM"].isna()]
if len(unmatched):
    print("Unmatched eligible players:", ", ".join(f"{p} ({s})" for p, s in unmatched[["PLAYER", "SEASON_ID"]].values))

# Players DARKO doesn't list (almost all deep-bench) get 0 = league-average impact
df["DPM"] = df["DPM"].fillna(0)
print(f"DPM filled with 0: {(df['DPM'] == 0).mean():.1%} of all rows")

df[df["PLAYER"].isin(["Nikola Jokić", "Luka Dončić"])][["PLAYER", "SEASON_ID", "DPM"]].tail(6)

## Create parque for players

In [ ]:
import time
from pathlib import Path

import pandas as pd
from nba_api.stats.endpoints import teamplayeronoffsummary

TOP_N = 25          # candidates per season
SLEEP_SECONDS = 2   # buffer between API calls

def season_label(season_id):          # 2024 -> "2023-24"
    return f"{season_id - 1}-{str(season_id)[-2:]}"

df["PLAYER_ID"] = df["PLAYER_ID"].astype(str)

# 1) Candidate filter: top 25 by VORP within each season
candidates = (
    df.dropna(subset=["VORP", "TEAM_ID"])
      .sort_values(["SEASON_ID", "VORP"], ascending=[True, False])
      .groupby("SEASON_ID")
      .head(TOP_N)
)

# 2) One API call per (season, team) that contains at least one candidate
pairs = candidates[["SEASON_ID", "TEAM_ID"]].drop_duplicates().astype(int).values.tolist()
print(f"{len(candidates)} candidates -> {len(pairs)} API calls (~{len(pairs) * SLEEP_SECONDS / 60:.0f} min)")

rows = []
for season_id, team_id in pairs:
    for attempt in range(3):
        try:
            res = teamplayeronoffsummary.TeamPlayerOnOffSummary(
                team_id=team_id,
                season=season_label(season_id),
                measure_type_detailed_defense="Advanced",
                timeout=60,
            )
            on, off = res.get_data_frames()[1], res.get_data_frames()[2]  # on-court, off-court tables
            break
        except Exception as e:
            print(f"retry {attempt + 1}: {season_id} team {team_id}: {e}")
            time.sleep(10 * (attempt + 1))
    else:
        print(f"SKIPPED {season_id} team {team_id}")
        continue

    m = on[["VS_PLAYER_ID", "NET_RATING"]].merge(
        off[["VS_PLAYER_ID", "NET_RATING"]], on="VS_PLAYER_ID", suffixes=("_ON", "_OFF")
    )
    m["SEASON_ID"] = season_id
    m["on_off_net_rtg"] = m["NET_RATING_ON"] - m["NET_RATING_OFF"]
    m["PLAYER_ID"] = m["VS_PLAYER_ID"].astype(str)
    m["TEAM_ID"] = team_id
    rows.append(m[["SEASON_ID", "PLAYER_ID", "TEAM_ID", "on_off_net_rtg"]])
    time.sleep(SLEEP_SECONDS)

onoff = pd.concat(rows, ignore_index=True)
onoff["TEAM_ID"] = onoff["TEAM_ID"].astype(df["TEAM_ID"].dtype)

# 3) Merge back; keep only candidates' values, 0 for everyone else.
# Join on TEAM_ID too: a player traded between two teams that both had a candidate comes back in
# both team responses, and joining on (SEASON_ID, PLAYER_ID) alone duplicated that player-season.
# TEAM_ID is the player's listed (latest) team, so traded candidates get that stint's on/off.
cand_keys = candidates[["SEASON_ID", "PLAYER_ID"]].assign(is_candidate=True)
df = (
    df.drop(columns=["on_off_net_rtg", "is_candidate"], errors="ignore")
      .merge(onoff, on=["SEASON_ID", "PLAYER_ID", "TEAM_ID"], how="left")
      .merge(cand_keys, on=["SEASON_ID", "PLAYER_ID"], how="left")
)
df["is_candidate"] = df["is_candidate"].fillna(False).astype(bool)
df.loc[~df["is_candidate"], "on_off_net_rtg"] = None   # drop non-candidate values pulled in by team calls
df["on_off_net_rtg"] = df["on_off_net_rtg"].fillna(0)
assert not df.duplicated(["SEASON_ID", "PLAYER_ID"]).any(), "duplicate player-seasons after on/off merge"

# 4) Save
out = Path("data/processed/model2_final_features.parquet")
out.parent.mkdir(parents=True, exist_ok=True)
df.to_parquet(out, index=False)
print(df.shape, "->", out)
df.loc[df["is_candidate"], ["SEASON_ID", "PLAYER", "VORP", "on_off_net_rtg"]].head()


# Building & Testing Model

In [ ]:
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
from xgboost import XGBRegressor

df = pd.read_parquet("data/processed/model2_final_features.parquet")
df = df.drop(columns=["is_candidate"], errors="ignore")

# Z_DPM: DARKO DPM Z-scored within each season against the rotation pool (same baseline as the other Z_ features)
pool = df[df["IN_REF_POOL"] & df["HAS_ADV"]].groupby("SEASON_ID")["DPM"]
df["Z_DPM"] = (df["DPM"] - df["SEASON_ID"].map(pool.mean())) / df["SEASON_ID"].map(pool.std(ddof=0))

FEATURES = [
    "Z_PTS", "Z_REB", "Z_AST", "Z_TS_PCT", "Z_USG_PCT",
    "Z_BPM", "Z_VORP", "Z_WS", "Z_WS_48", "Z_TEAM_WIN_PCT", "Z_CARRY_PROXY",
    "Z_DPM", "on_off_net_rtg",
]
TARGET = "MVP_SHARE"

data = df[df["HAS_ADV"]].dropna(subset=FEATURES).reset_index(drop=True)
train = data[data["SEASON_ID"] < 2024]
test = data[data["SEASON_ID"] >= 2024].copy()


def make_models():
    rf = RandomForestRegressor(n_estimators=500, min_samples_leaf=5, n_jobs=-1, random_state=0)
    xgb = XGBRegressor(n_estimators=300, max_depth=3, learning_rate=0.05, subsample=0.8,
                       colsample_bytree=0.8, min_child_weight=5, random_state=0)
    return rf, xgb


def ensemble_predict(rf, xgb, X):
    return np.clip((rf.predict(X) + xgb.predict(X)) / 2, 0, 1)


# ---- Evaluate: train on < 2024, test on >= 2024
rf, xgb = make_models()
rf.fit(train[FEATURES], train[TARGET])
xgb.fit(train[FEATURES], train[TARGET])
test["PRED_RF"] = np.clip(rf.predict(test[FEATURES]), 0, 1)
test["PRED_XGB"] = np.clip(xgb.predict(test[FEATURES]), 0, 1)
test["PRED_ENS"] = ensemble_predict(rf, xgb, test[FEATURES])

eligible = test[(test["MPG"] >= 20) & (test["GP"] >= test["MIN_GAMES_REQ"])]

summary = []
for col, name in [("PRED_RF", "RandomForest"), ("PRED_XGB", "XGBoost"), ("PRED_ENS", "Ensemble")]:
    hits = overlap = zero_vote = 0
    for season, s in eligible.groupby("SEASON_ID"):
        top5 = s.nlargest(5, col)
        hits += top5.iloc[0]["PLAYER_ID"] == s.loc[s[TARGET].idxmax(), "PLAYER_ID"]
        overlap += len(set(top5["PLAYER_ID"]) & set(s.nlargest(5, TARGET)["PLAYER_ID"]))
        zero_vote += (top5[TARGET] == 0).sum()
    summary.append({
        "model": name,
        "test_rmse_all": mean_squared_error(test[TARGET], test[col]) ** 0.5,
        "test_rmse_eligible": mean_squared_error(eligible[TARGET], eligible[col]) ** 0.5,
        "winners_correct": f"{hits}/3",
        "top5_overlap": f"{overlap}/15",
        "zero_vote_in_top5": zero_vote,
    })

for season, s in eligible.groupby("SEASON_ID"):
    print(f"\n{s['SEASON'].iloc[0]}  (actual MVP: {s.loc[s[TARGET].idxmax(), 'PLAYER']})")
    print(s.nlargest(5, "PRED_ENS")[["PLAYER", "TEAM", "PRED_RF", "PRED_XGB", "PRED_ENS", TARGET]]
          .round(3).to_string(index=False))

summary_df = pd.DataFrame(summary).set_index("model").round(4)
print("\nEnsemble vs. components (test seasons 2024-2026):")
print(summary_df.to_string())

# ---- Save for live predictions: refit on ALL labelled seasons (2014-2026) so the
# saved models have seen the most recent voting. The test metrics above come from the
# < 2024 fit; this refit has no held-out score of its own.
rf_live, xgb_live = make_models()
rf_live.fit(data[FEATURES], data[TARGET])
xgb_live.fit(data[FEATURES], data[TARGET])

out_dir = Path("models")
out_dir.mkdir(exist_ok=True)
bundle = {
    "rf": rf_live,
    "xgb": xgb_live,
    "features": FEATURES,
    "target": TARGET,
    "train_seasons": sorted(data["SEASON_ID"].unique().tolist()),
    "eval_metrics_2024_2026": summary_df.reset_index().to_dict("records"),
    "notes": "Prediction = clip((rf.predict(X) + xgb.predict(X)) / 2, 0, 1). "
             "Z_DPM = DPM Z-scored per season against IN_REF_POOL & HAS_ADV players.",
}
joblib.dump(bundle, out_dir / "model2_ensemble.joblib")
print(f"\nSaved -> {out_dir / 'model2_ensemble.joblib'} (trained on {bundle['train_seasons'][0]}-{bundle['train_seasons'][-1]})")
